In [ ]:
import sys
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
import shutil

# Add project root to path to import config
project_root = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(project_root))

from config import YOLO_DATA_DIR
print(f"YOLO data directory: {YOLO_DATA_DIR}")

In [ ]:
# Paths
images_path = YOLO_DATA_DIR / "images"
labels_path = YOLO_DATA_DIR / "labels"

train_img_path = YOLO_DATA_DIR / "train" / "images"
train_lbl_path = YOLO_DATA_DIR / "train" / "labels"
test_img_path  = YOLO_DATA_DIR / "test" / "images"
test_lbl_path  = YOLO_DATA_DIR / "test" / "labels"

print(f"Source images: {images_path}")
print(f"Train images: {train_img_path}")
print(f"Test images: {test_img_path}")

In [ ]:
# Create folders
for p in [train_img_path, train_lbl_path, test_img_path, test_lbl_path]:
    p.mkdir(parents=True, exist_ok=True)

# Collect images and labels
image_files = list(images_path.glob("*.jpg"))
labels_files = [labels_path/f"{img.stem}.txt" for img in image_files]

# Get class for stratification
# Assumes each image has a single object, so one line in YOLO txt
classes = []
for lbl_file in labels_files:
    with open(lbl_file) as f:
        line = f.readline().strip()
        cls_id = int(line.split()[0])  # first number in YOLO format is class
        classes.append(cls_id)
        
print(f"Found {len(image_files)} images")
print(f"Classes distribution: {set(classes)}")

In [5]:
# Stratified split
splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(image_files, classes))

# Copy files to train/test folders
for idx in train_idx:
    shutil.copy(image_files[idx], train_img_path / image_files[idx].name)
    shutil.copy(labels_files[idx], train_lbl_path / labels_files[idx].name)

for idx in test_idx:
    shutil.copy(image_files[idx], test_img_path / image_files[idx].name)
    shutil.copy(labels_files[idx], test_lbl_path / labels_files[idx].name)

print("Train/Test split done!")

Train/Test split done!
